In [76]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))


In [77]:
import glob
import pandas as pd
import re
from scipy.stats import t
from uvv.uvv_collection import UVVCollection
from uvv.uvv_batchcollection import UVVBatchCollection

In [78]:
def get_batch_from_identifier(identifier):
    """Extract batch name from entry identifier.
    Example: '20250728_uvv_alla_ln310006a_1' -> 'ln310006a'
    """
    parts = identifier.split('_')
    return parts[-2] if len(parts) >= 2 else identifier

def ends_with_a_or_a_digit(batch_name):
    """Check if batch ends with 'a' or 'a' followed by digit(s).
    Examples: 'ln310006a' -> True, 'ln310006a2' -> True, 'ln310006' -> False
    """
    return bool(re.search(r'a\d*$', batch_name))

def get_base_batch(batch_name):
    """Remove 'a' or 'a#' suffix to get base batch name.
    Examples: 'ln310006a' -> 'ln310006', 'ln310006a2' -> 'ln310006'
    """
    return re.sub(r'a\d*$', '', batch_name)

In [ ]:
input_folder = 'data/evaluation/single_batch/monophasic/'
files = glob.glob(input_folder + '*.csv')
files[0:5]

In [ ]:
package_dir = 'data/evaluation/single_batch/monophasic/'
evalcol = UVVBatchCollection.from_local(package_dir)
evalcol

In [ ]:
package_dir = 'data/processed/'
uvcoll = UVVCollection.from_local(package_dir).remove_repeated()
uvcoll_repeated = UVVCollection.from_local(package_dir)
print(len(uvcoll))
print(len(uvcoll_repeated))

# Defining the materials, wavelengths, loadings, sonications, and dates you want to use as filter criteria

In [82]:
filter_criteria = uvcoll.get_filter_criteria()
filter_criteria

{'materials': ['LN33',
  'LN31_1',
  'LN28_2',
  'BLPHI4',
  'LN28_1',
  'LN55',
  'LN46',
  'LN31',
  'LN31_2',
  'BLPHI',
  'LN28',
  'LN28SBA_1'],
 'wavelengths': [450, 365, 406],
 'loadings': [0.75, 1.0, 2.0, 1.5, 0.25, 0.5, '73.5h'],
 'sonications': [None, 10],
 'phases': ['aqueous']}

In [ ]:
import re
from collections import defaultdict
import yaml
import os

"""BATCH FILTERING SECTION

This section separates entries into two groups:
1. Normal batches: Batches that do NOT contain rerun variants.
2. Special batches: Groups of batches containing the original and its 'a' or 'a#' rerun variants.

The batch name is extracted from the YAML files in the raw folder (system.batch field).
"""

normal_entries = []
special_entries_dict = defaultdict(list)
grouped_entries = defaultdict(list)

# Extract batch names from YAML files
def get_batch_from_yaml(identifier):
    """Extract batch name from corresponding YAML file in the raw folder."""
    yaml_path = f'data/raw/{identifier}.yaml'
    if os.path.exists(yaml_path):
        with open(yaml_path, 'r') as f:
            data = yaml.safe_load(f)
            return data.get('system', {}).get('batch', identifier)
    return identifier

# Group all entries by their batch from YAML
for entry in evalcol:
    batch_name = get_batch_from_yaml(entry.identifier)
    base_batch = get_base_batch(batch_name)
    grouped_entries[base_batch].append((entry, batch_name))

# Separate groups into normal and special dictionaries
for base_batch, entries_with_batch in grouped_entries.items():
    is_special = len(entries_with_batch) > 1 or any(
        ends_with_a_or_a_digit(batch) for _, batch in entries_with_batch
    )
    
    entries = [entry for entry, _ in entries_with_batch]
    
    if is_special:
        special_entries_dict[base_batch] = entries
    else:
        normal_entries.extend(entries)

special_entries_dict = dict(special_entries_dict)

print(f"Normal batches: {len(normal_entries)}")
print(f"Special batch groups: {len(special_entries_dict)}")
print(f"Special batch groups: {list(special_entries_dict.keys())}")

In [84]:
"""NORMAL BATCHES: Add baseline calculation for entries that have 'c(H2O2)' column.

This loop:
1. Checks if each entry has 'c(H2O2)' column with no null values
2. Calculates baseline-adjusted H2O2 concentration: c(H2O2)_baseline = c(H2O2) - baseline
3. Creates combined_df containing only entries with successfully calculated baseline
"""

for entry in normal_entries:
    print(entry.df)
    if 'c(H2O2)' in entry.df.columns:
        if entry.df['c(H2O2)'].notnull().all():
            entry.df['c(H2O2)_baseline'] = entry.df['c(H2O2)'] - entry.baseline
        else:
            print(f"Entry {entry.identifier} has missing values in 'c(H2O2)' column.")
    else:
        print(f"Entry {entry.identifier} does not have the required columns or attributes.")

# Combined dataframe for normal batches only
combined_df = pd.concat([entry.df for entry in normal_entries if 'c(H2O2)_baseline' in entry.df.columns], ignore_index=True)
print(f"\nCombined DataFrame shape: {combined_df.shape}")
print(combined_df)

                        identifier  time    abs420  dilutionfactor  \
0    20240229_uvv_alla_blphi0001_1   0.0  0.089272             2.0   
1    20240229_uvv_alla_blphi0001_2   1.0  0.389035             2.0   
2    20240229_uvv_alla_blphi0001_3   2.0  0.477749             2.0   
3    20240229_uvv_alla_blphi0001_4   3.0  0.446437             2.0   
4    20240229_uvv_alla_blphi0001_5   4.0  0.433074             2.0   
5    20240229_uvv_alla_blphi0001_6   6.0  0.413520             2.0   
6    20240229_uvv_alla_blphi0001_7  24.0  0.408581             2.0   
7    20240229_uvv_alla_blphi0002_1   0.0  0.014858             2.0   
8    20240229_uvv_alla_blphi0002_2   1.0  0.329365             2.0   
9    20240229_uvv_alla_blphi0002_3   2.0  0.394820             2.0   
10   20240229_uvv_alla_blphi0002_4   3.0  0.409775             2.0   
11   20240229_uvv_alla_blphi0002_5   4.0  0.400719             2.0   
12   20240229_uvv_alla_blphi0002_6   6.0  0.433698             2.0   
13   20240229_uvv_al

In [ ]:
from scipy.stats import t, norm

"""NORMAL BATCHES: Group by material properties and calculate statistics.

This section:
1. Groups normal entries by material properties (material_name, wavelength, loading, etc.)
2. For each group, performs Grubbs outlier detection per time point
3. Calculates mean, std error, and 95% confidence interval
4. Saves results to: data/evaluation/material_excwavelength_loading_sonication/{key}.csv
"""
def norm_material(name):
    return 'BLPHI4' if str(name).upper().startswith('BLPHI') else name

grouped = combined_df.groupby(['material_name', 'exc_wavelength', 'loading', 'sonication', 'phase', 'synthesis_date_of_material'])
if 'sonication' in combined_df.columns:
    combined_df['sonication'] = combined_df['sonication'].fillna(0).astype(int)

separated_dfs = {}
for (material, wavelength, loading, sonication, phase, date), group in grouped:
    material = norm_material(material)
    key = f"{material}_{wavelength}nm_{str(loading).replace('.', '-')}_g_L_{str(sonication).replace('.','-')}min_{(phase)}_{date}"
    separated_dfs[key] = group.reset_index(drop=True)

for key, df in separated_dfs.items():
    times = pd.unique(df['time'])
    meanlist = []
    stdlist = []
    std_err_list = []
    measurements = []
    
    for time in times:
        def grubbs_test(data, alpha=0.05):
            n = len(data)
            mean = data.mean()
            std_dev = data.std()
            if std_dev != 0:
                G = max(abs(data - mean)) / std_dev
            else:
                G = 0
            t_critical = t.ppf(1 - alpha / (2 * n), n - 2)
            G_critical = ((n - 1) / (n ** 0.5)) * ((t_critical ** 2) / (n - 2 + t_critical ** 2)) ** 0.5
            return G, G_critical

        values = df[df.time == time]['c(H2O2)_baseline']
        outliers = []
        while True:
            G, G_critical = grubbs_test(values)
            if G > G_critical:
                outlier = values[abs(values - values.mean()).idxmax()]
                outliers.append(outlier)
                values = values[values != outlier]
            else:
                break
        df.loc[df.time == time, 'Grubbs test'] = df.loc[df.time == time, 'c(H2O2)_baseline'].apply(lambda x: 'FALSE' if x in outliers else 'TRUE')
        
        meanlist.append(values.mean())
        n = len(values)
        measurements.append(n)
        if n > 1:
            std_err = values.std() / (n ** 0.5)
            std_err_list.append(std_err)
            confidence_interval = t.ppf(1 - 0.05 / 2, n - 1) * std_err
            stdlist.append(confidence_interval)
        else:
            stdlist.append(float('nan'))
            std_err_list.append(float('nan'))

    print(f"Mean list for {key}: {meanlist}")
    data = {'times': times, 'average_c': meanlist, '95% confidence interval': stdlist, 'standard error': std_err_list, 'number of datapoints': measurements}
    result_df = pd.DataFrame(data)
    df.to_csv(f"data/evaluation/material_excwavelength_loading_sonication/{key}_calculatedfrom.csv", index=False)
    result_df.to_csv(f"data/evaluation/material_excwavelength_loading_sonication/{key}.csv", index=False)

## Make an evaluation ignoring the synthesis date

In [ ]:
# Normalizing material names first, then grouping
# Normalizing material names, treating any name starting with "BLPHI" as "BLPHI4" 
def norm_material(name):
    return 'BLPHI4' if str(name).upper().startswith('BLPHI') else name

grouped = (
    combined_df
    .assign(material_name=combined_df['material_name'].apply(norm_material))
    .groupby(['material_name', 'exc_wavelength', 'loading', 'sonication', 'phase'])
)

if 'sonication' in combined_df.columns:
    #print("There is a sonication column.")
    combined_df['sonication'] = combined_df['sonication'].fillna(0).astype(int)
    #if 10 in combined_df['sonication']:
        #print(combined_df['sonication'])
      
# Create a dictionary to store the separated DataFrames
separated_dfs = {}
# Iterate through the groups and store each group as a separate DataFrame
for (material, wavelength, loading, sonication, phase), group in grouped:
    normed_material = norm_material(material)
    key = f"{normed_material}_{wavelength}nm_{str(loading).replace('.', '-')}_g_L_{str(sonication).replace('.','-')}min_{(phase)}"
    separated_dfs[key] = group.reset_index(drop=True)
for key, df in separated_dfs.items():
    wavelength = int(df['exc_wavelength'].iloc[0])  # Extract wavelength
    loading = df['loading'].iloc[0]  # Extract loading
    
for key, df in separated_dfs.items():
    times = pd.unique(df['time'])
    meanlist = []
    stdlist = []
    std_err_list = []
    measurements = []
    for time in times:
        def grubbs_test(data, alpha=0.05):
            n = len(data)
            mean = data.mean()
            std_dev = data.std()
            if std_dev != 0:
                G = max(abs(data - mean)) / std_dev
            else:
                G = 0  # Consider the value valid if std_dev is 0
            t_critical = t.ppf(1 - alpha / (2 * n), n - 2)
            G_critical = ((n - 1) / (n ** 0.5)) * ((t_critical ** 2) / (n - 2 + t_critical ** 2)) ** 0.5
            return G, G_critical

        values = df[df.time == time]['c(H2O2)_baseline']
        outliers = []
        while True:
            G, G_critical = grubbs_test(values)
            if G > G_critical:
                outlier = values[abs(values - values.mean()).idxmax()]
                outliers.append(outlier)
                values = values[values != outlier]
            else:
                break
        df.loc[df.time == time, 'Grubbs test'] = df.loc[df.time == time, 'c(H2O2)_baseline'].apply(lambda x: 'FALSE' if x in outliers else 'TRUE')
        
        meanlist.append(values.mean())
        n = len(values)
        measurements.append(n)
        if n > 1:
            std_err = values.std() / (n ** 0.5)
            std_err_list.append(std_err)
            confidence_interval = t.ppf(1 - 0.05 / 2, n - 1) * std_err  # 95% confidence interval
            stdlist.append(confidence_interval)
        else:
            stdlist.append(float('nan'))
            std_err_list.append(float('nan'))

    print(f"Mean list for {key}: {meanlist}")
    print(f"Standard deviation list for {key}: {stdlist}")
    print(f"DataFrame for {key}:\n{df}")
    data = {'times': times, 'average_c': meanlist, '95% confidence interval': stdlist, 'standard error': std_err_list, 'number of datapoints': measurements}
    result_df = pd.DataFrame(data)
    result_df
    wavelength = int(df['exc_wavelength'].iloc[0])  # Extract wavelength
    loading = df['loading'].iloc[0]  # Extract loading
    df.to_csv(f"data/evaluation/material_excwavelength_loading_sonication/ignore_synthesis_date/{key}_calculatedfrom.csv", index=False)
    result_df.to_csv(f"data/evaluation/material_excwavelength_loading_sonication/ignore_synthesis_date/{key}.csv", index=False)


# Treating rerun batches (first ading baseline, then creating plots)

In [102]:
for base_batch, entries in special_entries_dict.items():
    for entry in entries:
        print(f"Processing entry {entry.identifier} in special batch group {base_batch}")
        if 'c(H2O2)' in entry.df.columns:
            if entry.df['c(H2O2)'].notnull().all():
                entry.df['c(H2O2)_baseline'] = entry.df['c(H2O2)'] - entry.baseline
            else:
                print(f"Entry {entry.identifier} has missing values in 'c(H2O2)' column.")
                print(base_batch, entry.identifier)
        else:
            print(f"Entry {entry.identifier} does not have required columns.")
            print(base_batch, entry.identifier)

Processing entry ln310006 in special batch group ln310006
Processing entry ln310006a in special batch group ln310006
Processing entry ln310008 in special batch group ln310008
Processing entry ln310008a in special batch group ln310008
Processing entry ln31_10015 in special batch group ln31_10015
Processing entry ln31_10015a in special batch group ln31_10015
Processing entry ln31_10016 in special batch group ln31_10016
Processing entry ln31_10016a in special batch group ln31_10016
Processing entry ln330033 in special batch group ln330033
Processing entry ln330033a in special batch group ln330033
Processing entry ln330033a2 in special batch group ln330033
Baseline value was not recorded at time = 0.0
Processing entry ln330033a3 in special batch group ln330033
Processing entry ln330034 in special batch group ln330034
Processing entry ln330034a in special batch group ln330034
Processing entry ln330034a2 in special batch group ln330034
Processing entry ln330034a3 in special batch group ln330

In [ ]:
"""SPECIAL BATCHES: Simplified evaluation with activity comparison.

This section:
1. For each group of special batches (e.g., ln310006 with its ln310006a variants):
    - Concatenates all time and c(H2O2)_baseline data
    - Normalizes by baseline (time == 0.0) for each batch
2. Creates a bar plot comparing original batch vs rerun variants (a, a2, etc.)
3. Uses material color for first cycle, colormap for reruns, and material_label_map for legend
4. Saves concatenated data to CSV for reference
"""

import matplotlib.pyplot as plt
from matplotlib import colormaps

# Define material colors and labels
material_colors = {
    'BLPHI4': 'orange',
    'LN55': 'red',
    'LN33': 'blue',
    'LN31': 'green',
    'LN46': 'black'
}

material_labels = {
    'BLPHI4': 'KPHI-lit',
    'LN55': 'KPHIS-1',
    'LN33': 'KPHIS-2',
    'LN31': 'KPHIS-3',
    'LN46': 'KPHI-b'
}

for base_batch, special_entries in special_entries_dict.items():
    # Find if base batch exists in normal_entries
    base_batch_entries = [e for e in normal_entries if get_batch_from_identifier(e.identifier) == base_batch]
    
    # Combine base batch entries with special variants
    all_entries_for_group = base_batch_entries + special_entries
    
    if not all_entries_for_group:
        continue
    
    # Create combined dataframe for this batch group
    rerun_df = pd.concat(
        [entry.df for entry in all_entries_for_group if 'c(H2O2)_baseline' in entry.df.columns],
        ignore_index=True
    )
    
    if rerun_df.empty:
        continue
    
    # Group by material_name and loading only (ignoring date and wavelength)
    if 'sonication' in rerun_df.columns:
        rerun_df['sonication'] = rerun_df['sonication'].fillna(0).astype(int)
    
    grouped_rerun = rerun_df.groupby(['material_name', 'loading', 'sonication', 'phase'])
    
    for (material, loading, sonication, phase), group in grouped_rerun:
        # Extract batch cycle number (original=0, a=1, a2=2, etc.)
        def extract_cycle(identifier):
            # Extract batch name from identifier (e.g., ln310006a from 20250728_uvv_alla_ln310006a_3)
            parts = identifier.split('_')
            batch_name = parts[-2] if len(parts) > 1 else identifier
            
            if batch_name.endswith('a3'):
                return 3
            elif batch_name.endswith('a2'):
                return 2
            elif batch_name.endswith('a'):
                return 1
            else:
                return 0  # original batch
        
        group = group.copy()
        group['batch_cycle'] = group['identifier'].apply(extract_cycle)
        
        # Normalize by baseline (time == 0.0) for each batch
        normalized_data = []
        for identifier in group['identifier'].unique():
            batch_data = group[group['identifier'] == identifier].copy()
            baseline_value = batch_data[batch_data['time'] == 0.0]['c(H2O2)_baseline'].values
            
            if len(baseline_value) > 0:
                baseline = baseline_value[0]
                batch_data['activity'] = batch_data['c(H2O2)_baseline'] - baseline
            else:
                batch_data['activity'] = batch_data['c(H2O2)_baseline']
            
            normalized_data.append(batch_data)
        
        combined_rerun_df = pd.concat(normalized_data, ignore_index=True)
        
        # Create bar plot
        fig, ax = plt.subplots(figsize=(14, 6))
        
        unique_times = sorted(combined_rerun_df[combined_rerun_df['time'] > 0]['time'].unique())
        cycles = sorted(combined_rerun_df['batch_cycle'].unique())
        bar_width = 0.15
        
        # Get material color and label from maps
        material_color = material_colors.get(material, 'gray')
        material_label = material_labels.get(material, material)
        
        # Colormap for rerun cycles (using matplotlib.colormaps to avoid deprecation)
        if len(cycles) > 1:
            colormap = colormaps['Set3']
            rerun_colors = [colormap(i / (len(cycles) - 1)) for i in range(len(cycles) - 1)]
        
        for idx, cycle in enumerate(cycles):
            cycle_data = combined_rerun_df[combined_rerun_df['batch_cycle'] == cycle]
            cycle_means = []
            
            for t in unique_times:
                time_data = cycle_data[cycle_data['time'] == t]['activity']
                cycle_means.append(time_data.mean())
            
            x_positions = [x + idx * bar_width for x in range(len(unique_times))]
            
            # First cycle uses material color, others use colormap
            if cycle == 0:
                bar_color = material_color
                label = material_label
            else:
                bar_color = rerun_colors[idx - 1]
                label = f'rerun_{chr(96 + cycle)}'
            
            ax.bar(x_positions, cycle_means, bar_width, label=label, color=bar_color)
        
        ax.set_xlabel('time / h', fontsize=12)
        ax.set_ylabel(r'c($H_2O_2$)', fontsize=12)
        ax.set_title(f'{material_label} - {str(loading).replace(".", "-")} g/L - {str(sonication).replace(".", "-")} min sonication', fontsize=14, fontweight='bold')
        ax.set_xticks([x + bar_width * (len(cycles) - 1) / 2 for x in range(len(unique_times))])
        ax.set_xticklabels([f'{t}' for t in unique_times])
        ax.legend(fontsize=10, title=material_label, title_fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        
        # Save plot with base_batch name
        plot_path = f"data/evaluation/material_excwavelength_loading_sonication/rerun/{base_batch.upper()}.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        # Save combined data with original measurements
        combined_rerun_df.to_csv(
            f"data/evaluation/material_excwavelength_loading_sonication/rerun/{base_batch.upper()}_data.csv", 
            index=False
        )
        print(f"Saved plot and data for {base_batch.upper()}")